# TIMIT Forced-Alignment Degradation Experiment

Tests whether forced alignment breaks when the pronunciation dictionary
does not exactly match what is spoken.  Three conditions per utterance:

| condition | lexicon |
|-----------|--------|
| canonical | TIMIT global dictionary (rarely matches realisation) |
| oracle    | per-file phones extracted from `.phn`/`.wrd` ground truth |
| degraded  | oracle with N random phone edits (N = 1 … 20) |

Aligners: **MFA** (TIMIT acoustic model, TIMIT phone set) and **Charsiu** (IPA).
Failure = aligner produces no output for the file.

## 1. Install dependencies

In [ ]:
import subprocess, sys

# MFA via conda (Kaggle ships conda)
subprocess.run(
    ["conda", "install", "-y", "-c", "conda-forge", "montreal-forced-aligner"],
    check=True,
)

# Charsiu via pip
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "charsiu"], check=True)

In [ ]:
# Download TIMIT acoustic model for MFA
!mfa model download acoustic timit

## 2. Paths

Adjust `TIMIT_ROOT` to point at your Kaggle dataset mount.
The TIMIT dataset on Kaggle typically contains a `TIMIT/` directory
with `TRAIN/` and `TEST/` subdirectories, and a `timit.dic` lexicon.

In [ ]:
import os
from pathlib import Path

TIMIT_ROOT = Path("/kaggle/input/timit-the-timit-acoustic-phonetic-continuous-speech/TIMIT")
TIMIT_LEXICON = TIMIT_ROOT / "DOC" / "timitdic.txt"  # adjust if path differs
OUTPUT_CSV = Path("/kaggle/working/results.csv")

# Quick sanity check
for p in [TIMIT_ROOT, TIMIT_LEXICON]:
    print(p, "exists:", p.exists())

## 3. Copy experiment scripts into working directory

In [ ]:
# If you uploaded the .py files as dataset attachments, copy them here.
# If running from a repo, this cell can be skipped.
import shutil

for script in ["timit_utils.py", "error_introduction.py",
               "alignment_runners.py", "run_experiment.py"]:
    src = Path("/kaggle/input/timit-experiment-scripts") / script
    if src.exists():
        shutil.copy(src, Path("/kaggle/working") / script)
        print("copied", script)

## 4. Smoke test on 10 files

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

from timit_utils import extract_file_lexicon, iter_timit_files, get_word_sequence

for wav, phn, wrd in list(iter_timit_files(TIMIT_ROOT))[:3]:
    lexicon = extract_file_lexicon(phn, wrd)
    words = get_word_sequence(wrd)
    print(wav.stem, words)
    for w, prons in lexicon.items():
        print(f"  {w}: {prons}")

In [ ]:
from run_experiment import run_all

SMOKE_CSV = Path("/kaggle/working/smoke_results.csv")
run_all(
    timit_root=TIMIT_ROOT,
    timit_lexicon_path=TIMIT_LEXICON,
    output_csv=SMOKE_CSV,
    n_trials_per_n_edits=2,
    max_edits=5,
    max_files=10,
)

## 5. Full run

In [ ]:
run_all(
    timit_root=TIMIT_ROOT,
    timit_lexicon_path=TIMIT_LEXICON,
    output_csv=OUTPUT_CSV,
    n_trials_per_n_edits=10,
    max_edits=20,
)

## 6. Analysis

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
df.head()

In [ ]:
# Failure rate by aligner and condition (canonical vs oracle)
base = df[df["condition"].isin(["canonical", "oracle"])]
print(base.groupby(["aligner", "condition"])["success"].mean())

In [ ]:
import matplotlib.pyplot as plt

degraded = df[df["condition"] == "degraded"]
pivot = (
    degraded.groupby(["aligner", "n_edits"])["success"]
    .mean()
    .unstack("aligner")
)

fig, ax = plt.subplots(figsize=(9, 5))
for aligner in pivot.columns:
    ax.plot(pivot.index, 1 - pivot[aligner], marker="o", label=aligner)

ax.set_xlabel("Number of random phone edits")
ax.set_ylabel("Alignment failure rate")
ax.set_title("Alignment failure vs. number of lexicon edits")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/failure_curve.png", dpi=150)
plt.show()

In [ ]:
# Summary table: failure rate per n_edits per aligner
table = (
    degraded.groupby(["n_edits", "aligner"])["success"]
    .agg(failure_rate=lambda x: 1 - x.mean(), n_trials="count")
    .reset_index()
)
print(table.to_string(index=False))